In [0]:
import logging
logging.disable(logging.CRITICAL)

In [0]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import random
import time
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


class LinkedInScraper:
    def __init__(self, keywords, location):
        self.keywords = keywords
        self.location = location
        self.base_url = (
            f"https://www.linkedin.com/jobs/search?"
            f"keywords={keywords}&location={location}&position=1&pageNum=0"
        )

    def get_job_ids(self):
        """Extract job IDs from search result page"""
        response = requests.get(self.base_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")
        jobs = soup.find_all("li")

        job_ids = []
        for job in jobs:
            try:
                base_card_div = job.find("div", {"class": "base-card"})
                job_id = base_card_div.get("data-entity-urn").split(":")[3]
                job_ids.append(job_id)
            except Exception:
                continue
        return job_ids

    def get_job_details(self, job_id):
        """Extract job details from job posting page"""
        job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"
        response = requests.get(job_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")
        job_post = {}

        def safe_extract(selector, multiple=False):
            try:
                if multiple:
                    return [s.text.strip() for s in soup.select(selector)]
                return soup.select_one(selector).text.strip()
            except Exception:
                return None

        # Base job info
        job_post["job_id"] = job_id
        job_post["job_title"] = safe_extract("h2.top-card-layout__title")
        job_post["company_name"] = safe_extract("a.topcard__org-name-link")
        job_post["time_posted"] = safe_extract("span.posted-time-ago__text")
        job_post["num_applicants"] = safe_extract("span.num-applicants__caption")
        job_post["location"] = self.location

        # Criteria
        criteria = safe_extract("span.description__job-criteria-text", multiple=True)
        if criteria and len(criteria) >= 4:
            job_post["seniority_level"] = criteria[0]
            job_post["employment_type"] = criteria[1]
            job_post["job_function"] = criteria[2]
            job_post["industries"] = criteria[3]
        else:
            job_post["seniority_level"] = None
            job_post["employment_type"] = None
            job_post["job_function"] = None
            job_post["industries"] = None

        return job_post

    def scrape_jobs(self):
        """Main method to scrape jobs"""
        job_ids = self.get_job_ids()
        job_list = []

        for job_id in job_ids:
            details = self.get_job_details(job_id)
            job_list.append(details)
            time.sleep(random.uniform(1, 3))  # polite scraping

        return job_list


In [0]:
from scraper import LinkedInScraper

locations = ["India", "USA", "Australia"]
keyword = "data engineer"

for loc in locations:
    scraper = LinkedInScraper(keyword, loc)
    print(f"🔍 {loc} → {scraper.base_url}")

In [0]:
job_ids = scraper.get_job_ids()
print("Extracted Job IDs:", job_ids)
print("Total Jobs Found:", len(job_ids))

if not job_ids:
    print(scraper.base_url)
    response = requests.get(scraper.base_url, headers=HEADERS)
    print(response.text[:1000])

In [0]:
if job_ids:
    sample_job_id = job_ids[2]
    print("Testing with Job ID:", sample_job_id)

    job_details = scraper.get_job_details(sample_job_id)
    print("Job Details Extracted:")
    print(job_details)